# 🧬 Подготовка белка (рецептора) к молекулярному докингу

Данный ноутбук реализует автоматизированный пайплайн подготовки кристаллической структуры белка-мишени к молекулярному докингу. 

### Основные этапы подготовки:
1. **Загрузка PDB-структуры:** Скачивание структуры с RCSB Protein Data Bank по её ID (или загрузка локального файла).
2. **Очистка структуры рецептора:**
   * Удаление молекул воды (`HOH`, `WAT`).
   * Удаление солей, растворителей и кристаллографических добавок (`SO4`, `CL`, `EDO`, `GOL`, `DMS`, `PEG`).
   * **Выделение сокристаллизованного лиганда** (например, Lenvatinib с кодом `L7A` или `LEV`) для последующего определения зоны связывания, с последующим его удалением из файла белка.
3. **Генерация параметров поискового бокса (`config.txt`):** Автоматический расчет геометрического центра и размеров активного центра на основе координат сокристаллизованного лиганда.
4. **Конвертация белка в формат PDBQT:** Подготовка структуры для работы AutoDock Vina с добавлением полярных водородов и частичных зарядов.

## 🛠️ Шаг 1: Импорт библиотек и настройка параметров

In [2]:
import os
import urllib.request
import numpy as np
import pandas as pd

# Параметры подготовки
PDB_ID = "3WZD"               # ID структуры в PDB (например, VEGFR2 в комплексе с Lenvatinib)
LIGAND_CODE = "LEV"           # Трехбуквенный код сокристаллизованного лиганда (Lenvatinib в 5ZHX имеет код L7A, также часто используется LEV или др.)
BUFFER_SIZE = 6.0            # Дополнительный отступ вокруг лиганда в ангстремах (Å) для размера поискового бокса

# Пути к файлам
WORK_DIR = "./prepared_protein"
os.makedirs(WORK_DIR, exist_ok=True)

raw_pdb_path = os.path.join(WORK_DIR, f"{PDB_ID}.pdb")
cleaned_protein_path = os.path.join(WORK_DIR, f"{PDB_ID}_cleaned.pdb")
extracted_ligand_path = os.path.join(WORK_DIR, f"{PDB_ID}_ligand.pdb")
config_path = os.path.join(WORK_DIR, "config.txt")
receptor_pdbqt_path = os.path.join(WORK_DIR, f"{PDB_ID}_cleaned.pdbqt")

## 📥 Шаг 2: Скачивание PDB-файла
Если у вас уже есть локальный файл, вы можете поместить его в папку `prepared_protein/` и пропустить этот шаг.

In [3]:
if not os.path.exists(raw_pdb_path):
    url = f"https://files.rcsb.org/download/{PDB_ID}.pdb"
    print(f"Скачивание структуры {PDB_ID} с RCSB PDB... ({url})")
    try:
        urllib.request.urlretrieve(url, raw_pdb_path)
        print(f"Успешно сохранено в {raw_pdb_path}")
    except Exception as e:
        print(f"Ошибка при скачивании: {e}")
else:
    print(f"Файл {raw_pdb_path} уже существует.")

Скачивание структуры 3WZD с RCSB PDB... (https://files.rcsb.org/download/3WZD.pdb)
Успешно сохранено в ./prepared_protein\3WZD.pdb


## 🧹 Шаг 3: Очистка структуры белка и извлечение лиганда
На этом этапе мы выполняем строковый парсинг PDB файла:
1. **Извлекаем лиганд**: Строки `HETATM` или `ATOM` с кодом `LIGAND_CODE` (например, `L7A` или `LEV`) сохраняются в отдельный файл. Это позволит нам узнать точное расположение сайта связывания.
2. **Очищаем белок**: Из белка удаляются все молекулы воды, кристаллизационные добавки и ионы, а также сам сокристаллизованный лиганд, чтобы освободить место для докируемых молекул.

In [4]:
def clean_protein_and_extract_ligand(input_pdb, out_protein_pdb, out_ligand_pdb, ligand_code):
    # Список частых кристаллизационных добавок и ионов для исключения
    ignored_residues = {
        "HOH", "WAT", "SO4", "CL", "EDO", "GOL", "DMS", "PEG", 
        "ACT", "NH4", "NA", "K", "MG", "PO4", "TRS", "FMT"
    }
    
    protein_lines = []
    ligand_lines = []
    
    removed_residues = set()
    
    with open(input_pdb, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                res_name = line[17:20].strip()
                
               # Если это нужный нам сокристаллизованный лиганд
                if res_name == ligand_code:
                    ligand_lines.append(line)
                # Если это вода или иные добавки
                elif res_name in ignored_residues:
                    removed_residues.add(res_name)
                    continue
                # Все остальные стандартные и нестандартные аминокислоты белка
                else:
                    protein_lines.append(line)
            elif line.startswith("TER"):
                protein_lines.append(line)
            elif line.startswith("END"):
                break
                
    # Сохраняем очищенный белок
    with open(out_protein_pdb, "w", encoding="utf-8") as f:
        f.writelines(protein_lines)
        f.write("END\n")
        
    # Сохраняем выделенный лиганд
    if ligand_lines:
        with open(out_ligand_pdb, "w", encoding="utf-8") as f:
            f.writelines(ligand_lines)
            f.write("END\n")
        print(f"✅ Успешно извлечен сокристаллизованный лиганд '{ligand_code}' ({len(ligand_lines)} атомов) -> {out_ligand_pdb}")
    else:
        print(f"⚠️ Предупреждение: лиганд с кодом '{ligand_code}' не найден в структуре. Проверьте правильность кода.")
        
    print(f"✅ Белок успешно очищен и сохранен -> {out_protein_pdb}")
    print(f"🗑️ Удалены молекулы добавок/воды: {', '.join(removed_residues) if removed_residues else 'Нет'}")

clean_protein_and_extract_ligand(raw_pdb_path, cleaned_protein_path, extracted_ligand_path, LIGAND_CODE)

✅ Успешно извлечен сокристаллизованный лиганд 'LEV' (30 атомов) -> ./prepared_protein\3WZD_ligand.pdb
✅ Белок успешно очищен и сохранен -> ./prepared_protein\3WZD_cleaned.pdb
🗑️ Удалены молекулы добавок/воды: GOL, SO4, EDO, HOH


## 📐 Шаг 4: Расчет геометрических параметров бокса поиска (`config.txt`)
Для работы AutoDock Vina крайне важно настроить параметры бокса (размеры и центр). 
Мы автоматизируем этот процесс, считывая координаты вырезанного сокристаллизованного лиганда. Центр бокса совпадет с центром масс лиганда, а размеры бокса будут равны габаритам лиганда с добавлением буфера `BUFFER_SIZE` (по умолчанию 10 Å).

In [9]:
def calculate_grid_box(ligand_pdb, output_config, receptor_name, buffer=10.0):
    x_coords = []
    y_coords = []
    z_coords = []
    
    if not os.path.exists(ligand_pdb):
        print("⚠️ Файл лиганда отсутствует, невозможно автоматически рассчитать бокс.")
        return
        
    with open(ligand_pdb, "r") as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                try:
                    x_coords.append(float(line[30:38].strip()))
                    y_coords.append(float(line[38:46].strip()))
                    z_coords.append(float(line[46:54].strip()))
                except ValueError:
                    continue
                    
    if not x_coords:
        print("⚠️ Координаты атомов лиганда не были найдены.")
        return
        
    # Вычисляем минимум, максимум и центр координат
    x_min, x_max = min(x_coords), max(x_coords)
    y_min, y_max = min(y_coords), max(y_coords)
    z_min, z_max = min(z_coords), max(z_coords)
    
    center_x = (x_max + x_min) / 2
    center_y = (y_max + y_min) / 2
    center_z = (z_max + z_min) / 2
    
    # Вычисляем размер бокса (разница координат + буфер)
    size_x = (x_max - x_min) + buffer
    size_y = (y_max - y_min) + buffer
    size_z = (z_max - z_min) + buffer
    
    # Запись config.txt
    config_content = f"""# Конфигурационный файл для AutoDock Vina
# Сгенерирован автоматически на основе положения лиганда {LIGAND_CODE}

receptor = {receptor_name}

center_x = {center_x:.3f}
center_y = {center_y:.3f}
center_z = {center_z:.3f}

size_x = {size_x:.1f}
size_y = {size_y:.1f}
size_z = {size_z:.1f}

num_modes = 9
energy_range = 3.0
exhaustiveness = 8
"""
    
    with open(output_config, "w", encoding="utf-8") as f:
        f.write(config_content)
        
    print("📊 Рассчитанные параметры активного центра:")
    print(f"   Центр: x={center_x:.3f}, y={center_y:.3f}, z={center_z:.3f}")
    print(f"   Размер бокса: x={size_x:.1f}, y={size_y:.1f}, z={size_z:.1f}")
    print(f"✅ Параметры успешно сохранены в: {output_config}")

calculate_grid_box(extracted_ligand_path, config_path, f"{PDB_ID}_cleaned.pdbqt", buffer=BUFFER_SIZE)

📊 Рассчитанные параметры активного центра:
   Центр: x=1.536, y=-6.661, z=15.338
   Размер бокса: x=11.7, y=12.9, z=21.8
✅ Параметры успешно сохранены в: ./prepared_protein\config.txt


## 🔄 Шаг 5: Конвертация белка в формат PDBQT
Формат PDBQT необходим Vina для правильного расчета сил электростатического и гидрофобного взаимодействия.

Конвертация осуществляется с помощью утилиты `prepare_receptor4.py` из пакета AutoDock Tools (MGLTools). В проекте предусмотрен встроенный Python-интерфейс в модуле `chemplus`.

### Вариант 1: Использование встроенного модуля `chemplus`

In [13]:
import importlib
from chemplus.vina import pdbqt
importlib.reload(pdbqt)

# Укажите путь к интерпретатору pythonsh от MGLTools на вашем компьютере
MGL_PYTHON_PATH = r"C:\Program Files (x86)\MGLTools-1.5.7\python.exe" # Измените путь под свою систему

if os.path.exists(MGL_PYTHON_PATH):
    try:
        print("Конвертация белка в PDBQT через chemplus...")
        conversion_result = pdbqt.pdb_to_pdbqt(
            cleaned_protein_path,
            receptor_pdbqt_path,
            mgl_python=MGL_PYTHON_PATH,
            timeout=1800
        )
        if conversion_result != 1 or not os.path.exists(receptor_pdbqt_path):
            raise RuntimeError(f"PDBQT conversion failed or timed out: result={conversion_result}, output={receptor_pdbqt_path}")
        print(f"✅ Файл рецептора PDBQT успешно создан: {receptor_pdbqt_path}")
    except Exception as e:
        print(f"❌ Ошибка конвертации: {e}")
else:
    print(f"⚠️ По указанному пути MGL_PYTHON_PATH ({MGL_PYTHON_PATH}) не найден MGLTools.")
    print("Пожалуйста, установите MGLTools или укажите корректный путь для выполнения конвертации.")

Конвертация белка в PDBQT через chemplus...
✅ Файл рецептора PDBQT успешно создан: ./prepared_protein\3WZD_cleaned.pdbqt


### Вариант 2: Запуск через системную консоль (прямой вызов скрипта из проекта)
В папке `Docking/chemnotes-main/Docking/Docking_Verification/` вашего проекта уже находится скрипт `prepare_receptor4.py`. 
Вы можете вызвать его напрямую через консоль Windows/Linux:

```bash
mgltools_python prepare_receptor4.py -r prepared_protein/5ZHX_cleaned.pdb -o prepared_protein/5ZHX_cleaned.pdbqt -A checkhydrogens
```

## 🎉 Подготовка завершена!
Теперь у вас есть готовый набор входных данных для проведения молекулярного докинга в папке `prepared_protein/`:
1. **`3WZD_cleaned.pdbqt`** — готовый к докингу протонированный белок без лишних молекул сокристаллизованного лиганда и воды.
2. **`config.txt`** — конфигурационный файл Vina с автоматически рассчитанными координатами и размерами бокса.
3. **`../pre-trained/protonated_prepared_passed_3D_2026-08-06_22-39.sdf`** (или другой сгенерированный SDF файл лигандов) — библиотека 3D-координат лигандов.

Вы можете передать эти файлы в функцию локального докинга или на суперкомпьютер!